# Experiment 10: Repetition Penalty Test & Resource Profiling

**Purpose:** Test the untested claim that rep_penalty=1.15 preserves RefPref while reducing Rep-4

**5 Conditions (all max_new_tokens=800, greedy):**
1. Baseline (no steering, no rep penalty)
2. Hard Cutoff K=16 — NO rep penalty
3. Hard Cutoff K=16 — WITH rep_penalty=1.15
4. Linear Decay K=16 — NO rep penalty
5. Linear Decay K=16 — WITH rep_penalty=1.15

**Logging:** Every 25 samples with ETA + checkpoint saves every 100 samples

In [ ]:
!pip install -q bitsandbytes accelerate transformers torch bert-score rouge-score tqdm
print('Dependencies installed!')

In [ ]:
import os, json, glob, random, time, math, gc, datetime
import numpy as np
import torch
from tqdm import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

BEST_LAYER = 8
ALPHA_0 = 18.0
K = 16
REP_PENALTY = 1.15
MAX_NEW_TOKENS = 800
LOG_EVERY = 25
CHECKPOINT_EVERY = 100

OUTPUT_DIR = '/kaggle/working'
print(f'Config: Layer={BEST_LAYER}, a0={ALPHA_0}, K={K}, rep_pen={REP_PENALTY}, tokens={MAX_NEW_TOKENS}')
print(f'Logging every {LOG_EVERY}, checkpoint every {CHECKPOINT_EVERY}')

In [ ]:
# Load Dataset
DATA_FILENAME = 'vietnamese_medical_halueval_15k_specialized.json'
search_paths = [
    f'/kaggle/input/**/{DATA_FILENAME}', f'/kaggle/input/{DATA_FILENAME}',
    f'data/{DATA_FILENAME}', f'./{DATA_FILENAME}',
    f'E:/Paper_Steering_VN_15K/data/{DATA_FILENAME}'
]
data_path = None
for p in search_paths:
    m = glob.glob(p, recursive=True)
    if m: data_path = m[0]; break
if not data_path: raise FileNotFoundError(f'{DATA_FILENAME} not found')
print(f'Dataset: {data_path}')

with open(data_path, 'r', encoding='utf-8') as f:
    raw_dataset = json.load(f)
shuffled_records = list(raw_dataset)
random.seed(SEED); random.shuffle(shuffled_records)
n_total = len(shuffled_records)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)
train_records = shuffled_records[:n_train]
test_records = shuffled_records[n_train + n_val:]
test_subset = test_records[:500]
print(f'Train: {len(train_records)}, Test: {len(test_records)}, Eval: {len(test_subset)}')

In [ ]:
# Load Model + Resource Profile
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
print(f'Loading {MODEL_NAME}...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
peak_mem_load = torch.cuda.max_memory_allocated()/(1024**3) if torch.cuda.is_available() else 0
print(f'Model loaded! Params: {total_params:,} ({total_params/1e9:.2f}B)')
print(f'Peak GPU after load: {peak_mem_load:.2f} GB')
print(f'Hook target: model.model.layers[{BEST_LAYER}]')
print(f'Hidden dim: {model.config.hidden_size}, Num layers: {model.config.num_hidden_layers}')

In [ ]:
# Compute Steering Vector
PROMPT_TEMPLATE = """Dua vao ngu canh y hoc sau day, hay tra loi cau hoi:
Ngu canh: {context}
Cau hoi: {question}
Tra loi: """

probe_count = min(400, len(train_records))
probe_records = train_records[:probe_count]
print(f'Extracting Layer {BEST_LAYER} activations from {probe_count} pairs...')
activations_pos, activations_neg = [], []
t0 = time.time()

with torch.no_grad():
    for i, rec in enumerate(probe_records):
        ctx = rec.get('knowledge_context', rec.get('context', ''))
        q = rec['question']
        text_pos = PROMPT_TEMPLATE.format(context=ctx, question=q) + rec['right_answer']
        text_neg = PROMPT_TEMPLATE.format(context=ctx, question=q) + rec['hallucinated_answer']
        inp_p = tokenizer(text_pos, return_tensors='pt', padding=False).to('cuda')
        inp_n = tokenizer(text_neg, return_tensors='pt', padding=False).to('cuda')
        out_p = model(**inp_p, output_hidden_states=True)
        out_n = model(**inp_n, output_hidden_states=True)
        activations_pos.append(out_p.hidden_states[BEST_LAYER][0,-1,:].detach().float().cpu().numpy())
        activations_neg.append(out_n.hidden_states[BEST_LAYER][0,-1,:].detach().float().cpu().numpy())
        if (i+1) % 50 == 0:
            el = time.time()-t0; eta = el/(i+1)*(probe_count-i-1)
            print(f'  Extraction: {i+1}/{probe_count} | {el/60:.1f}min elapsed | ETA {eta/60:.1f}min')

v_raw = np.mean(activations_pos, axis=0) - np.mean(activations_neg, axis=0)
v_steer_np = v_raw / np.linalg.norm(v_raw)
v_steer = torch.tensor(v_steer_np, dtype=torch.bfloat16, device='cuda')
print(f'v_steer computed in {(time.time()-t0)/60:.1f}min, dim={v_steer.shape[0]}')

In [ ]:
# === EVALUATION ENGINE WITH FULL LOGGING ===
from bert_score import score as bert_score_fn

class SteeringHook:
    def __init__(self, layer_idx, v_vector, alpha=18.0, K=16, decay='hard'):
        self.layer_idx = layer_idx
        self.v_vector = v_vector
        self.alpha = alpha
        self.K = K
        self.decay = decay
        self.step_counter = 0
        self.handle = None
    def _eff_alpha(self, t):
        if self.K >= 999: return self.alpha
        if t >= self.K: return 0.0
        if self.decay == 'hard': return self.alpha
        elif self.decay == 'linear': return self.alpha * (1.0 - t / self.K)
        return self.alpha
    def hook_fn(self, module, inputs, output):
        a = self._eff_alpha(self.step_counter)
        if a > 0:
            if isinstance(output, tuple):
                h = output[0]
                v = self.v_vector.to(h.device).to(h.dtype)
                h[:, -1, :] = h[:, -1, :] + a * v
                output = (h,) + output[1:]
            else:
                v = self.v_vector.to(output.device).to(output.dtype)
                output[:, -1, :] = output[:, -1, :] + a * v
        self.step_counter += 1
        return output
    def register(self, mdl):
        self.step_counter = 0
        self.handle = mdl.model.layers[self.layer_idx].register_forward_hook(self.hook_fn)
    def remove(self):
        if self.handle: self.handle.remove(); self.handle = None

def run_eval(model, tokenizer, records, v_vector, alpha, K, decay,
             max_new_tokens, rep_penalty=1.0, layer_idx=8, name=''):
    gen_texts, ref_ans, neg_ans = [], [], []
    latencies, token_counts, eos_hits, rep4_scores = [], [], [], []
    total_n = len(records)
    t_start = time.time()
    
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    
    print(f'\n{"="*70}')
    print(f'[{name}] START: {total_n} samples, max_tokens={max_new_tokens}, rep_pen={rep_penalty}')
    print(f'[{name}] alpha={alpha}, K={K}, decay={decay}, layer={layer_idx}')
    if torch.cuda.is_available():
        print(f'[{name}] GPU: {torch.cuda.memory_allocated()/(1024**3):.2f} GB')
    print(f'[{name}] Time: {datetime.datetime.now().strftime("%H:%M:%S")}')
    print(f'{"="*70}')
    
    for idx, rec in enumerate(records):
        ctx = rec.get('knowledge_context', rec.get('context', ''))
        q = rec['question']
        prompt = PROMPT_TEMPLATE.format(context=ctx, question=q)
        inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
        
        hook = None
        if v_vector is not None:
            hook = SteeringHook(layer_idx, v_vector, alpha=alpha, K=K, decay=decay)
            hook.register(model)
        
        t0 = time.time()
        with torch.no_grad():
            out_ids = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                     do_sample=False, temperature=1.0, top_p=1.0,
                                     repetition_penalty=rep_penalty)
        lat = time.time() - t0
        if hook: hook.remove()
        
        gen_tok = out_ids[0][inputs.input_ids.shape[1]:]
        gen_text = tokenizer.decode(gen_tok, skip_special_tokens=True)
        
        # EOS check
        eos_id = tokenizer.eos_token_id
        hit_eos = bool(len(gen_tok) > 0 and gen_tok[-1].item() == eos_id)
        
        # Rep-4
        words = gen_text.split()
        if len(words) >= 4:
            ngrams = [tuple(words[i:i+4]) for i in range(len(words)-3)]
            rep4 = (1.0 - len(set(ngrams))/len(ngrams)) * 100 if ngrams else 0.0
        else: rep4 = 0.0
        
        gen_texts.append(gen_text)
        ref_ans.append(rec['right_answer'])
        neg_ans.append(rec['hallucinated_answer'])
        latencies.append(lat)
        token_counts.append(len(gen_tok))
        eos_hits.append(hit_eos)
        rep4_scores.append(rep4)
        
        # Detailed log
        if (idx+1) % LOG_EVERY == 0 or idx == total_n - 1:
            elapsed = time.time() - t_start
            speed = (idx+1) / elapsed
            eta = (total_n - idx - 1) / speed if speed > 0 else 0
            gpu_gb = torch.cuda.memory_allocated()/(1024**3) if torch.cuda.is_available() else 0
            r_eos = np.mean(eos_hits) * 100
            r_rep4 = np.mean(rep4_scores)
            print(f'  [{name}] {idx+1:>4}/{total_n} | '
                  f'{elapsed/60:.1f}m | ETA {eta/60:.1f}m | '
                  f'{speed:.2f}s/s | lat {np.mean(latencies[-LOG_EVERY:]):.2f}s | '
                  f'tok {np.mean(token_counts[-LOG_EVERY:]):.0f} | '
                  f'EOS {r_eos:.1f}% | Rep4 {r_rep4:.2f}% | GPU {gpu_gb:.2f}GB')
        
        # Checkpoint
        if (idx+1) % CHECKPOINT_EVERY == 0:
            safe = name.replace(' ', '_').replace('/', '_')
            ckpt = {'name': name, 'done': idx+1, 'total': total_n,
                    'texts': gen_texts.copy(), 'refs': ref_ans.copy(), 'negs': neg_ans.copy(),
                    'eos_hits': eos_hits.copy(), 'rep4s': rep4_scores.copy()}
            ckpt_path = os.path.join(OUTPUT_DIR, f'ckpt_{safe}_{idx+1}.json')
            with open(ckpt_path, 'w', encoding='utf-8') as f:
                json.dump(ckpt, f, ensure_ascii=False)
            print(f'  [{name}] >>> CHECKPOINT: {ckpt_path}')
    
    peak_mem = torch.cuda.max_memory_allocated()/(1024**3) if torch.cuda.is_available() else 0
    gen_time = time.time() - t_start
    print(f'\n  [{name}] Generation done: {gen_time/60:.1f}min ({gen_time/total_n:.2f}s/sample)')
    print(f'  [{name}] Peak GPU: {peak_mem:.2f} GB')
    
    # BERTScore
    print(f'  [{name}] Computing BERTScore (verbose)...')
    t_bs = time.time()
    _, _, bs_ref = bert_score_fn(gen_texts, ref_ans, lang='vi',
                                  model_type='bert-base-multilingual-cased', num_layers=9,
                                  verbose=True, batch_size=64)
    _, _, bs_neg = bert_score_fn(gen_texts, neg_ans, lang='vi',
                                  model_type='bert-base-multilingual-cased', num_layers=9,
                                  verbose=True, batch_size=64)
    bs_time = time.time() - t_bs
    print(f'  [{name}] BERTScore done: {bs_time/60:.1f}min')
    
    bs_r, bs_n = bs_ref.numpy(), bs_neg.numpy()
    rp = (bs_r > bs_n).astype(int)
    
    result = {
        'name': name,
        'refpref_pct': float(rp.mean()*100), 'refpref_count': int(rp.sum()), 'total': len(rp),
        'bertscore_f1_mean': float(np.mean(bs_r)), 'bertscore_f1_std': float(np.std(bs_r)),
        'rep4_mean': float(np.mean(rep4_scores)), 'rep4_std': float(np.std(rep4_scores)),
        'eos_pct': float(np.mean(eos_hits)*100), 'eos_count': int(sum(eos_hits)),
        'lat_mean': float(np.mean(latencies)), 'lat_std': float(np.std(latencies)),
        'tok_mean': float(np.mean(token_counts)),
        'peak_gpu_gb': peak_mem, 'rep_penalty': rep_penalty,
        'gen_time_min': gen_time/60, 'bs_time_min': bs_time/60
    }
    
    print(f'\n  ========== RESULT [{name}] ==========')
    print(f'  RefPref:    {result["refpref_pct"]:.2f}% ({result["refpref_count"]}/{result["total"]})')
    print(f'  BERTScore:  {result["bertscore_f1_mean"]:.4f} +/- {result["bertscore_f1_std"]:.4f}')
    print(f'  Rep-4:      {result["rep4_mean"]:.2f}% +/- {result["rep4_std"]:.2f}%')
    print(f'  EOS Hit:    {result["eos_pct"]:.1f}% ({result["eos_count"]}/{result["total"]})')
    print(f'  Latency:    {result["lat_mean"]:.2f}s +/- {result["lat_std"]:.2f}s')
    print(f'  Avg Tokens: {result["tok_mean"]:.1f}')
    print(f'  Peak GPU:   {result["peak_gpu_gb"]:.2f} GB')
    print(f'  Total time: {(gen_time+bs_time)/60:.1f}min')
    print(f'  ======================================')
    return result

print('Engine ready (detailed logging + checkpoints).')

In [ ]:
# === 10A: Baseline (no steering, no rep penalty) ===
print('\n' + '#'*70)
print('# 10A: UNSTEERED BASELINE')
print('#'*70)
r_baseline = run_eval(model, tokenizer, test_subset,
    v_vector=None, alpha=0, K=0, decay='hard',
    max_new_tokens=MAX_NEW_TOKENS, rep_penalty=1.0,
    layer_idx=BEST_LAYER, name='Baseline')
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# === 10B: Hard Cutoff — NO rep penalty ===
print('\n' + '#'*70)
print('# 10B: HARD CUTOFF (K=16) — NO REP PENALTY')
print('#'*70)
r_hard_no = run_eval(model, tokenizer, test_subset,
    v_vector=v_steer, alpha=ALPHA_0, K=K, decay='hard',
    max_new_tokens=MAX_NEW_TOKENS, rep_penalty=1.0,
    layer_idx=BEST_LAYER, name='HardCutoff_NoPen')
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# === 10C: Hard Cutoff — WITH rep_penalty=1.15 ===
print('\n' + '#'*70)
print(f'# 10C: HARD CUTOFF (K=16) — REP PENALTY={REP_PENALTY}')
print('#'*70)
r_hard_rep = run_eval(model, tokenizer, test_subset,
    v_vector=v_steer, alpha=ALPHA_0, K=K, decay='hard',
    max_new_tokens=MAX_NEW_TOKENS, rep_penalty=REP_PENALTY,
    layer_idx=BEST_LAYER, name='HardCutoff_RepPen')
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# === 10D: Linear Decay — NO rep penalty ===
print('\n' + '#'*70)
print('# 10D: LINEAR DECAY (K=16) — NO REP PENALTY')
print('#'*70)
r_linear_no = run_eval(model, tokenizer, test_subset,
    v_vector=v_steer, alpha=ALPHA_0, K=K, decay='linear',
    max_new_tokens=MAX_NEW_TOKENS, rep_penalty=1.0,
    layer_idx=BEST_LAYER, name='LinearDecay_NoPen')
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# === 10E: Linear Decay — WITH rep_penalty=1.15 ===
print('\n' + '#'*70)
print(f'# 10E: LINEAR DECAY (K=16) — REP PENALTY={REP_PENALTY}')
print('#'*70)
r_linear_rep = run_eval(model, tokenizer, test_subset,
    v_vector=v_steer, alpha=ALPHA_0, K=K, decay='linear',
    max_new_tokens=MAX_NEW_TOKENS, rep_penalty=REP_PENALTY,
    layer_idx=BEST_LAYER, name='LinearDecay_RepPen')
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# === FINAL COMPARISON TABLE ===
all_r = [r_baseline, r_hard_no, r_hard_rep, r_linear_no, r_linear_rep]

print('\n' + '='*110)
print('EXPERIMENT 10 — FINAL COMPARISON TABLE')
print('='*110)
print(f'{"Condition":<25} {"RefPref%":<10} {"BS-F1":<10} {"Rep4%":<10} {"EOS%":<8} {"Lat(s)":<10} {"Tokens":<8} {"GPU(GB)":<8} {"Time(m)":<8}')
print('-'*110)
for r in all_r:
    tt = r.get('gen_time_min',0) + r.get('bs_time_min',0)
    print(f'{r["name"]:<25} {r["refpref_pct"]:<10.2f} {r["bertscore_f1_mean"]:<10.4f} '
          f'{r["rep4_mean"]:<10.2f} {r["eos_pct"]:<8.1f} {r["lat_mean"]:<10.2f} '
          f'{r["tok_mean"]:<8.1f} {r["peak_gpu_gb"]:<8.2f} {tt:<8.1f}')
print('='*110)

# Key comparisons
print(f'\n--- HARD CUTOFF: Effect of rep_penalty={REP_PENALTY} ---')
print(f'  RefPref: {r_hard_no["refpref_pct"]:.2f}% -> {r_hard_rep["refpref_pct"]:.2f}% (delta: {r_hard_rep["refpref_pct"]-r_hard_no["refpref_pct"]:+.2f}pp)')
print(f'  Rep-4:   {r_hard_no["rep4_mean"]:.2f}% -> {r_hard_rep["rep4_mean"]:.2f}% (delta: {r_hard_rep["rep4_mean"]-r_hard_no["rep4_mean"]:+.2f}pp)')
print(f'  BS-F1:   {r_hard_no["bertscore_f1_mean"]:.4f} -> {r_hard_rep["bertscore_f1_mean"]:.4f}')

print(f'\n--- LINEAR DECAY: Effect of rep_penalty={REP_PENALTY} ---')
print(f'  RefPref: {r_linear_no["refpref_pct"]:.2f}% -> {r_linear_rep["refpref_pct"]:.2f}% (delta: {r_linear_rep["refpref_pct"]-r_linear_no["refpref_pct"]:+.2f}pp)')
print(f'  Rep-4:   {r_linear_no["rep4_mean"]:.2f}% -> {r_linear_rep["rep4_mean"]:.2f}% (delta: {r_linear_rep["rep4_mean"]-r_linear_no["rep4_mean"]:+.2f}pp)')
print(f'  BS-F1:   {r_linear_no["bertscore_f1_mean"]:.4f} -> {r_linear_rep["bertscore_f1_mean"]:.4f}')

# Verdict
hard_rep4_down = r_hard_rep['rep4_mean'] < r_hard_no['rep4_mean']
hard_rp_ok = abs(r_hard_rep['refpref_pct'] - r_hard_no['refpref_pct']) < 3.0
print(f'\nVERDICT (Hard Cutoff):')
print(f'  Rep-4 reduced: {hard_rep4_down} ({r_hard_no["rep4_mean"]:.2f}% -> {r_hard_rep["rep4_mean"]:.2f}%)')
print(f'  RefPref preserved (<3pp change): {hard_rp_ok}')

# Resource Profile
print(f'\n--- RESOURCE PROFILE ---')
print(f'  Model: {MODEL_NAME}')
print(f'  Total params: {total_params:,} ({total_params/1e9:.2f}B)')
print(f'  Quantization: 4-bit NF4 double-quant, bfloat16 compute')
print(f'  Peak GPU (model load): {peak_mem_load:.2f} GB')
print(f'  Peak GPU (generation): {max(r["peak_gpu_gb"] for r in all_r):.2f} GB')
print(f'  Hook: model.model.layers[{BEST_LAYER}].register_forward_hook()')
print(f'  Modified tensor: output[0][:, -1, :] (last token position)')

# Save
summary = {
    'experiment': 'Experiment_10_RepPenalty_ResourceProfile',
    'config': {'layer': BEST_LAYER, 'alpha_0': ALPHA_0, 'K': K,
               'rep_penalty': REP_PENALTY, 'max_new_tokens': MAX_NEW_TOKENS, 'n_test': len(test_subset)},
    'results': {r['name']: {k:v for k,v in r.items() if k != 'ref_pref_array'} for r in all_r},
    'resource_profile': {
        'model': MODEL_NAME, 'total_params': total_params, 'params_B': total_params/1e9,
        'quantization': '4bit-NF4-double', 'peak_gpu_load_gb': peak_mem_load,
        'peak_gpu_gen_gb': max(r['peak_gpu_gb'] for r in all_r),
        'hook_module': f'model.model.layers[{BEST_LAYER}]',
        'hidden_dim': model.config.hidden_size, 'num_layers': model.config.num_hidden_layers
    },
    'verdict': {
        'hard_rep4_reduced': hard_rep4_down, 'hard_refpref_preserved': hard_rp_ok,
        'hard_rep4_delta': r_hard_rep['rep4_mean'] - r_hard_no['rep4_mean'],
        'hard_refpref_delta': r_hard_rep['refpref_pct'] - r_hard_no['refpref_pct']
    }
}
out_path = os.path.join(OUTPUT_DIR, 'experiment_10_rep_penalty_results.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f'\nSaved: {out_path}')
print(f'Completed at: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')